# Reasoning-trace segment labeler (Qwen3.5-27B GGUF + llama.cpp, 2 x Tesla T4, data parallel)

Labels every `⟦S#⟧` segment in `thinking_text` with `discourse_role`, `temporal_role`, `answer_status` and `answer_value`.

**Before running:** turn on *GPU T4 x2* and *Internet*, put your CSV in `/kaggle/working/`, set `CFG.INPUT_FILENAME` in cell 2. Then run the cells top to bottom.

| Cell | What it does |
|---|---|
| 1-2 | Helpers and configuration (everything you may want to change is in `CFG`) |
| 3-4 | Build `llama-server` with CUDA (slow once, then reused) and download the GGUF (one file, shared by both GPUs) |
| 5 | Load the CSV and split every trace into its existing segments |
| 6 | **The prompt**: a short system prompt + one worked example (sent as a real user/assistant turn) |
| 7-8 | Labeling logic, then checkpoint and CSV export |
| 9-10 | Start **one `llama-server` per GPU** (each T4 holds its own full copy of the model) and label a few sample traces so you can eyeball quality before the long run |
| 11 | Full run. Traces come from **one shared queue, served by both GPUs**. Checkpoint line after **every trace**, **both CSV files rewritten every 5 minutes** |
| 12 | Summary: counts, label distribution, failures |

**How the two T4s are used (data parallelism)**
- Cell 9 starts two independent `llama-server` processes, one per T4 (`CUDA_VISIBLE_DEVICES` pins each process to its own GPU). Every GPU holds a full copy of the model, so there is no GPU-to-GPU traffic.
- Cell 11 puts all traces in one shared queue. Every server has `SERVER_SLOTS` worker threads that take the next trace whenever they are free, so the two GPUs stay balanced automatically.
- If one server dies, its workers stop and the other GPU finishes the remaining traces. Nothing finished is lost.
- A 27B model does not fit one T4 at 4-bit (Q4_K_M is 16.7 GB, a T4 has 15 GB), so the default is the 3-bit `UD-IQ3_XXS` (11.5 GB). It leaves room for the KV cache. Change `CFG.GGUF_QUANT` in cell 2 to try another quant (see the comment there).

**What changed compared with the first version**
- The 28,000-character skill is gone. The model now gets a ~700-word system prompt and one worked example, so its attention is not buried under 8k tokens of rules.
- The model writes a one-line `note` for each segment *before* its labels (a tiny bit of reasoning inside the JSON), so it does not have to jump straight to the label.
- The whole trace is always shown (long traces only shorten the *context* segments, never the ones being labeled). No windowing planner, no benchmark, no validation report.

**Resuming:** if you only stopped cell 11 (kernel and servers still alive), just run cell 11 again. If the session or kernel restarted, run cells 1-10 again and then cell 11. Finished traces are skipped. Changing the prompt, the example, the model or `CHUNK_SIZE` invalidates old results automatically (they are relabeled), so checkpoint lines from the first version are simply ignored.

**Speed knobs and the throughput report**
- `CFG.SERVER_SLOTS` (cell 2) = requests each GPU decodes at the same time. Pick it from the free VRAM that cell 9 prints (the comment in cell 2 explains how). `SERVER_SLOTS` and `CTX_PER_SLOT` are not part of the checkpoint fingerprint, so changing them keeps old results valid. After changing them, run `stop_servers()` and cell 9 again.
- Cell 10 labels its sample traces concurrently on all servers and prints the wall-clock time.
- Cells 11 and 12 end with a throughput report (traces per minute, prompt and completion tokens/s, overall and per GPU) so you can compare settings.

**Outputs** (in `CFG.OUTPUT_DIR`): `labeled_results.csv` (original columns + `rtl_*` columns), `labeled_segments.csv` (one row per segment, with the model's one-line `note`), `labeling_checkpoint.jsonl`.

In [1]:
# 1. Imports and small helpers
import subprocess, sys, os, json, time, shutil, re, hashlib, threading, random, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED


def sh(cmd, timeout=None):
    """Run a command (list) and capture its output. A missing binary is reported as a failed command, not a crash."""
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except FileNotFoundError as e:
        return subprocess.CompletedProcess(cmd, 127, "", str(e))


def md5(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()

In [2]:
# 2. Configuration: every setting lives here
from types import SimpleNamespace

CFG = SimpleNamespace(
    # ---- input / output ----------------------------------------------------------------------
    INPUT_DIR="/kaggle/working/",
    INPUT_FILENAME="r1-llama8b_raw_results_segmented.csv",   # <-- your CSV
    OUTPUT_DIR="/kaggle/working/",
    TRACE_COL="thinking_text",                    # column holding the segmented trace (⟦S1⟧ text ⟦S2⟧ text ...)
    QUESTION_COL=None,                            # optional column with the question text (shown to the model as context)
    TRACE_ID_COLS=["benchmark", "compute_level", "question_id"],   # copied into labeled_segments.csv (those that exist)
    OUT_RESULTS="labeled_results.csv",
    OUT_SEGMENTS="labeled_segments.csv",
    OUT_CHECKPOINT="labeling_checkpoint.jsonl",
    CSV_ENCODING="utf-8-sig",                     # utf-8-sig = Excel shows ⟦S1⟧ correctly. Read it back with encoding="utf-8-sig"
    AUTOSAVE_SECONDS=300,                         # rewrite the output CSVs every N seconds during the run
    LIMIT=None,                                   # e.g. 20 to label only the first 20 parseable traces (quick test)

    # ---- model and llama.cpp -----------------------------------------------------------------
    GGUF_REPO_ID="unsloth/Qwen3.5-9B-GGUF",
    GGUF_FILENAME="Qwen3.5-9B-Q4_K_M.gguf",
    WORK_DIR="/kaggle/temp",                      # heavy files (llama.cpp build, model); not saved with the notebook
    LLAMA_CPP_REPO="https://github.com/ggml-org/llama.cpp",
    LLAMA_CPP_REF="master",                       # branch or tag

    # ---- server ------------------------------------------------------------------------------
    GPU_INDICES=[0, 1],                           # one llama-server per entry, each pinned to its own GPU (data parallelism)
    PORT_BASE=8089,                               # GPU number k in the list listens on PORT_BASE + k
    # ---- concurrency: THE speed knob ---------------------------------------------------------
    # SERVER_SLOTS = requests each GPU decodes at the same time = worker threads per GPU in cells 10 and 11
    # (parallel requests in total = len(GPU_INDICES) x SERVER_SLOTS). One request leaves most of a T4 idle (decoding is
    # limited by memory bandwidth), so a few requests decoded together cost little more than one: more slots = more
    # traces/minute, until VRAM runs out or the GPU is saturated (diminishing returns).
    #
    # WHAT IT COSTS: llama-server is started with  -c SERVER_SLOTS x CTX_PER_SLOT  and  --parallel SERVER_SLOTS (cell 9),
    # so the KV cache grows linearly with SERVER_SLOTS x CTX_PER_SLOT. Weights and compute buffers do not grow.
    #
    # HOW TO CHOOSE: start the servers and read the used / FREE VRAM that cell 9 prints for each GPU.
    #   - 9B Q4_K_M (weights ~5.3 GiB): more room than the 27B, but check first: with 2 slots a T4 was already ~12 of 15 GiB
    #     full in one run (weights + KV + compute buffer). Go 2 -> 3 -> 4 (-> 6) one step at a time while cell 9 shows > 1 GiB free.
    #   - 27B UD-IQ3_XXS (weights 11.5 GB): tight -> 2 (1 if cell 9 warns).
    #   - Keep at least VRAM_WARN_GIB (about 1 GiB) free: cell 9 warns below that. Running out of memory in the middle
    #     of a long trace is worse than a slower run.
    #   - To see what one slot costs:  !grep "buffer size" /kaggle/temp/llama-server-gpu0.log  (the KV / state
    #     buffers scale with SERVER_SLOTS; the weights and the compute buffer do not).
    #   - Out of VRAM but you want more slots? Lower CTX_PER_SLOT (cell 5 prints the longest trace; it must still fit
    #     in one slot), or keep KV_CACHE_TYPE="q8_0".
    # After changing SERVER_SLOTS or CTX_PER_SLOT: run stop_servers(), then cell 9 again (a running server keeps its old
    # setting). Neither is part of FINGERPRINT, so existing checkpoint lines stay valid.
    # Compare settings with the throughput report at the end of cell 11 / 12.
    SERVER_SLOTS=6,                               # requests decoded in parallel PER GPU (= client threads per GPU)
    CTX_PER_SLOT=16384,                           # tokens available to ONE request (prompt + output)
    VRAM_WARN_GIB=1.0,                            # cell 9 warns when a GPU has less free VRAM than this after the servers are up
    BATCH_SIZE=2048,
    UBATCH_SIZE=512,
    FLASH_ATTN=True,                              # needed for a quantized KV cache; also saves memory
    KV_CACHE_TYPE="q8_0",                         # "q8_0" halves the KV cache vs f16; None = f16 (needs more VRAM)

    # ---- labeling ----------------------------------------------------------------------------
    CHUNK_SIZE=20,                                # max segments labeled per request (long traces are split into chunks)
    USE_NOTE=True,                                # model writes a one-line note per segment before its labels (helps accuracy, ~20 extra tokens/segment)
    NOTE_MAX_CHARS=100,                           # None = no length limit in the JSON schema
    ENFORCE_RULES=True,                           # cheap consistency fixes, see enforce_rules() in cell 7
    TEMPERATURE=0.0,
    RETRY_TEMPERATURE=0.3,                        # retries must differ from the first attempt
    MAX_RETRIES=2,
    REQUEST_TIMEOUT_S=1800,                       # a 27B model on a T4 is slower than the 9B was
    SEED=1234,
    SAMPLE_TRACES=3,                              # traces labeled in cell 10 for a quick quality check (not saved)
)

for _d in (CFG.OUTPUT_DIR, CFG.WORK_DIR):
    os.makedirs(_d, exist_ok=True)
print(json.dumps(vars(CFG), indent=1, default=str))

{
 "INPUT_DIR": "/kaggle/working/",
 "INPUT_FILENAME": "r1-llama8b_raw_results_segmented.csv",
 "OUTPUT_DIR": "/kaggle/working/",
 "TRACE_COL": "thinking_text",
 "QUESTION_COL": null,
 "TRACE_ID_COLS": [
  "benchmark",
  "compute_level",
  "question_id"
 ],
 "OUT_RESULTS": "labeled_results.csv",
 "OUT_SEGMENTS": "labeled_segments.csv",
 "OUT_CHECKPOINT": "labeling_checkpoint.jsonl",
 "CSV_ENCODING": "utf-8-sig",
 "AUTOSAVE_SECONDS": 300,
 "LIMIT": null,
 "GGUF_REPO_ID": "unsloth/Qwen3.5-9B-GGUF",
 "GGUF_FILENAME": "Qwen3.5-9B-Q4_K_M.gguf",
 "WORK_DIR": "/kaggle/temp",
 "LLAMA_CPP_REPO": "https://github.com/ggml-org/llama.cpp",
 "LLAMA_CPP_REF": "master",
 "GPU_INDICES": [
  0,
  1
 ],
 "PORT_BASE": 8089,
 "SERVER_SLOTS": 6,
 "CTX_PER_SLOT": 16384,
 "VRAM_WARN_GIB": 1.0,
 "BATCH_SIZE": 2048,
 "UBATCH_SIZE": 512,
 "FLASH_ATTN": true,
 "KV_CACHE_TYPE": "q8_0",
 "CHUNK_SIZE": 20,
 "USE_NOTE": true,
 "NOTE_MAX_CHARS": 100,
 "ENFORCE_RULES": true,
 "TEMPERATURE": 0.0,
 "RETRY_TEMPERATURE": 0

In [3]:
# 3. Packages and llama.cpp (built from source with CUDA; skipped when the binary already exists)
def pip_install(pkgs):
    r = sh([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    if r.returncode != 0:
        raise RuntimeError("pip install failed for " + " ".join(pkgs) + "\n" + r.stderr[-2000:])


pip_install(["pandas", "numpy", "requests", "huggingface_hub", "tqdm"])

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

r = sh(["nvidia-smi", "-L"])
print(r.stdout.strip() or r.stderr.strip())
if r.returncode != 0:
    raise RuntimeError("No GPU visible. Turn on a GPU (Tesla T4) in the Kaggle session settings.")
N_GPUS = sum(1 for _l in r.stdout.splitlines() if _l.startswith("GPU "))
if N_GPUS < len(CFG.GPU_INDICES) or max(CFG.GPU_INDICES) >= N_GPUS:
    raise RuntimeError(f"CFG.GPU_INDICES={CFG.GPU_INDICES} but nvidia-smi shows {N_GPUS} GPU(s). "
                       "On Kaggle choose Session options -> Accelerator -> GPU T4 x2, or edit CFG.GPU_INDICES.")

LLAMA_SRC = os.path.join(CFG.WORK_DIR, "llama.cpp")
LLAMA_BUILD = os.path.join(LLAMA_SRC, "build")
LLAMA_SERVER_BIN = os.path.join(LLAMA_BUILD, "bin", "llama-server")
CUDA_ROOT = "/usr/local/cuda"
# libcuda.so is the NVIDIA driver library; the CUDA::cuda_driver CMake target needs its path (the real driver first)
CUDA_DRIVER_CANDIDATES = [
    "/usr/local/nvidia/lib64/libcuda.so",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/local/nvidia/lib64/libcuda.so.1",
    os.path.join(CUDA_ROOT, "lib64", "stubs", "libcuda.so"),
    "/usr/local/cuda-12.8/compat/libcuda.so",
]


def find_cuda_driver_lib():
    for p in CUDA_DRIVER_CANDIDATES:
        if os.path.exists(p):
            return p
    raise RuntimeError("libcuda.so not found. Run  !find / -name 'libcuda.so*'  and add the path to CUDA_DRIVER_CANDIDATES.")


def cuda_arch():
    r = sh(["nvidia-smi", "-i", str(CFG.GPU_INDICES[0]), "--query-gpu=compute_cap", "--format=csv,noheader"])
    cap = r.stdout.strip().splitlines()[0].strip() if r.returncode == 0 and r.stdout.strip() else ""
    return cap.replace(".", "") if re.fullmatch(r"\d\.\d", cap) else "75"   # Tesla T4 = 7.5


def run_logged(cmd, log_path, heartbeat_s=60):
    """Run a long command, write its output to a log file, print a heartbeat, raise with the log tail on failure."""
    t0 = last = time.time()
    with open(log_path, "w") as lf:
        p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(2)
            if time.time() - last >= heartbeat_s:
                print(f"  ... still running ({int(time.time() - t0)} s): {' '.join(map(str, cmd))[:90]}")
                last = time.time()
    if p.returncode != 0:
        tail = "".join(open(log_path).readlines()[-60:])
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\nLast log lines ({log_path}):\n{tail}")


def ensure_llama_cpp():
    if os.path.isfile(LLAMA_SERVER_BIN):
        print("llama-server already built:", LLAMA_SERVER_BIN)
        return
    for tool in ("git", "cmake", "nvcc"):
        if shutil.which(tool) is None:
            raise RuntimeError(f"'{tool}' not found; building llama.cpp needs git, cmake and the CUDA toolkit (nvcc).")
    if not os.path.isdir(LLAMA_SRC):
        run_logged(["git", "clone", "--depth", "1", "--branch", CFG.LLAMA_CPP_REF, CFG.LLAMA_CPP_REPO, LLAMA_SRC],
                   os.path.join(CFG.WORK_DIR, "git_clone.log"))
    driver = find_cuda_driver_lib()
    arch = cuda_arch()
    print(f"Configuring llama.cpp (CUDA arch {arch}, driver lib {driver}) ...")
    run_logged(["cmake", "-S", LLAMA_SRC, "-B", LLAMA_BUILD, "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release",
                f"-DCMAKE_CUDA_ARCHITECTURES={arch}", "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF",
                f"-DCUDAToolkit_ROOT={CUDA_ROOT}", f"-DCUDA_cuda_driver_LIBRARY={driver}",
                f"-DCMAKE_LIBRARY_PATH={os.path.dirname(driver)}"],
               os.path.join(CFG.WORK_DIR, "cmake_configure.log"))
    print("Compiling llama-server (about 20-40 minutes on Kaggle's 4 CPU cores; an interrupted build resumes where it stopped) ...")
    run_logged(["cmake", "--build", LLAMA_BUILD, "--target", "llama-server", "-j", str(os.cpu_count() or 4)],
               os.path.join(CFG.WORK_DIR, "cmake_build.log"))
    if not os.path.isfile(LLAMA_SERVER_BIN):
        raise RuntimeError("Build finished but the llama-server binary is missing: " + LLAMA_SERVER_BIN)


ensure_llama_cpp()

GPU 0: Tesla T4 (UUID: GPU-ee6d9690-4c83-32ed-24ad-c165655d0232)
GPU 1: Tesla T4 (UUID: GPU-3db0a6ff-891d-c48e-f87f-0b87af7eb227)
Configuring llama.cpp (CUDA arch 75, driver lib /usr/local/nvidia/lib64/libcuda.so) ...
Compiling llama-server (about 20-40 minutes on Kaggle's 4 CPU cores; an interrupted build resumes where it stopped) ...
  ... still running (60 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (120 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (180 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (240 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (300 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (360 s): cmake --build /kaggle/temp/llama.cpp/build --target llama-server -j 4
  ... still running (420 s): cmake --build /kaggle/temp/llama.cpp/bui

In [5]:
# 4. Download the GGUF model (cached in WORK_DIR). ONE file on disk; both servers load it.
from huggingface_hub import hf_hub_download, list_repo_files

MODEL_DIR = os.path.join(CFG.WORK_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
HF_TOKEN = os.environ.get("HF_TOKEN") or None


def resolve_gguf_files():
    """Return the repo file(s) for CFG.GGUF_QUANT (several only when the quant is split into shards; the first one is loaded)."""
    if CFG.GGUF_FILENAME:
        return [CFG.GGUF_FILENAME]
    files = [f for f in list_repo_files(CFG.GGUF_REPO_ID, token=HF_TOKEN) if f.endswith(".gguf") and "mmproj" not in f.lower()]
    tag = re.escape(CFG.GGUF_QUANT.lower())
    hits = sorted(f for f in files if re.search(r"(?<![a-z0-9])" + tag + r"(?![a-z0-9_])", f.lower()))
    if not hits:
        raise ValueError(f"No .gguf file for GGUF_QUANT={CFG.GGUF_QUANT!r} in {CFG.GGUF_REPO_ID}. Files: {files}")
    shards = [f for f in hits if re.search(r"-\d{5}-of-\d{5}\.gguf$", f)]
    if shards:
        return shards
    if len(hits) > 1:
        print("Several files match, using the first:", hits)
    return hits[:1]


_files = resolve_gguf_files()
_paths = [hf_hub_download(CFG.GGUF_REPO_ID, _f, local_dir=MODEL_DIR, token=HF_TOKEN) for _f in _files]
GGUF_PATH = _paths[0]                                 # for a sharded quant llama.cpp finds the other shards next to the first one
_size_gib = sum(os.path.getsize(q) for q in _paths) / 2**30
print(f"GGUF: {GGUF_PATH}  ({_size_gib:.2f} GiB)")
if _size_gib > 12.8:
    print("WARNING: the weights alone take most of a 15 GB T4. If llama-server runs out of memory in cell 9, "
          "set SERVER_SLOTS=1, lower CTX_PER_SLOT, or pick a smaller GGUF_QUANT.")

Qwen3.5-9B-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 5.68GB            

Qwen3.5-9B-Q4_K_M.gguf: downloading bytes:           |  0.00B            

GGUF: /kaggle/temp/models/Qwen3.5-9B-Q4_K_M.gguf  (5.29 GiB)


In [6]:
# 5. Load the CSV and split every trace into its existing segments
INPUT_PATH = os.path.join(CFG.INPUT_DIR, CFG.INPUT_FILENAME)
if not os.path.isfile(INPUT_PATH):
    csvs = sorted(f for f in os.listdir(CFG.INPUT_DIR) if f.lower().endswith(".csv"))
    raise FileNotFoundError(f"{INPUT_PATH} not found. Set CFG.INPUT_FILENAME in cell 2. CSV files in {CFG.INPUT_DIR}: {csvs}")

DF = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8-sig")
DF = DF.head(len(DF) // 2).copy()
for _c in (CFG.TRACE_COL, CFG.QUESTION_COL):
    if _c is not None and _c not in DF.columns:
        raise KeyError(f"Column '{_c}' not in the CSV. Columns: {list(DF.columns)}")
ID_COLS = [c for c in CFG.TRACE_ID_COLS if c in DF.columns]

MARKER_RE = re.compile(r"(⟦S\d+⟧)")


def split_trace(text):
    """'⟦S1⟧ a ⟦S2⟧ b' -> [{'id': '⟦S1⟧', 'text': 'a'}, {'id': '⟦S2⟧', 'text': 'b'}]. Text before the first marker is ignored."""
    parts = MARKER_RE.split(text)          # [before, id1, text1, id2, text2, ...]
    return [{"id": parts[i], "text": parts[i + 1].strip()} for i in range(1, len(parts) - 1, 2)]


TRACES, SKIPPED = [], {}                    # SKIPPED: row_id -> reason (rows that are never sent to the model)
for row_id, row in enumerate(DF.to_dict("records")):
    text = row[CFG.TRACE_COL]
    segs = split_trace(text)
    if not segs:
        SKIPPED[row_id] = "no segment markers"
    elif len({s["id"] for s in segs}) != len(segs):
        SKIPPED[row_id] = "duplicate segment markers"
    else:
        TRACES.append({"row_id": row_id, "key": f"{row_id}:{md5(text)}", "ids": {c: row[c] for c in ID_COLS},
                       "question": row[CFG.QUESTION_COL] if CFG.QUESTION_COL else None, "segments": segs})
if CFG.LIMIT:
    for t in TRACES[CFG.LIMIT:]:
        SKIPPED[t["row_id"]] = "not selected (CFG.LIMIT)"
    TRACES = TRACES[:CFG.LIMIT]

n_segs = [len(t["segments"]) for t in TRACES]
longest = max((sum(len(s["text"]) for s in t["segments"]) for t in TRACES), default=0)
print(f"{INPUT_PATH}: {len(DF):,} rows, columns {list(DF.columns)}")
print(f"Traces to label: {len(TRACES):,}   skipped rows: {len(SKIPPED)} {dict(Counter(SKIPPED.values()))}")
if TRACES:
    print(f"Segments: {sum(n_segs):,} in total; per trace min {min(n_segs)}, median {int(statistics.median(n_segs))}, max {max(n_segs)}")
    print(f"Longest trace: ~{longest // 3:,} tokens (estimate); one request can hold {CFG.CTX_PER_SLOT:,} tokens including the prompt and the answer")

/kaggle/working/r1-llama8b_raw_results_segmented.csv: 4,725 rows, columns ['question_id', 'benchmark', 'difficulty_tier', 'task_type', 'compute_level', 'sample_index', 'raw_model_output', 'thinking_text', 'gold', 'is_correct', 'tokens_used', 'thinking_tokens', 'output_tokens', 'hit_response_marker', 'latency_s', 'was_truncated', 'hit_marker_empty_output']
Traces to label: 4,045   skipped rows: 680 {'no segment markers': 680}
Segments: 133,533 in total; per trace min 1, median 24, max 498
Longest trace: ~13,688 tokens (estimate); one request can hold 16,384 tokens including the prompt and the answer


In [7]:
# 6. The prompt: a short system prompt + ONE worked example, sent as a real user/assistant exchange
NOTE_RULE = ("`note` is one short line (what the model does in this segment, and whether the answer is reopened later). "
             "Write it first, then decide the labels from it.\n\n") if CFG.USE_NOTE else ""
OUT_KEYS = "segment_id, " + ("note, " if CFG.USE_NOTE else "") + "discourse_role, answer_status, answer_value, temporal_role"

SYSTEM_PROMPT = f"""You label the segments of an LLM's reasoning trace. The trace is already split into segments marked ⟦S1⟧, ⟦S2⟧, ... Never split, merge, rewrite or skip segments.

Label each segment by the FUNCTION it serves in the reasoning, judged in context: read what came before it and what comes after it. Never label by keywords: "wait", "let me check", "so the answer is" mean different things in different places. Ask: "given everything before it, what is the model doing in this segment?" If two labels seem plausible, pick the dominant function of the segment. Do not hedge.

{NOTE_RULE}discourse_role
- setup: reading the INPUT: restating the question, listing the givens, the answer options or available tools, defining a term from the question, re-reading the problem (also mid-trace), restating the goal. No fact or method is applied yet.
- reasoning: any step toward the answer: recalling a fact/formula/criterion TO APPLY it, announcing a plan, calculating, deducing, evaluating or rejecting an option, resolving a doubt, self-posed planning questions ("How do I implement this?"), switching approach with no suspected error, guess-and-test used to FIND a solution, writing code, stating a conclusion.
- doubt: the model voices UNCERTAINTY about something it holds ("I'm not sure", "maybe I misread", "is that right?", "that doesn't make sense"). Planning questions and approach switches are NOT doubt.
- verification: a check actually performed (or opened with "let me double-check") on something already derived: recomputing, substituting back, testing on samples, a second method. First-time computations and steps needed to proceed (unit conversion) are NOT verification, even if they say "make sure".
- correction: the model recognizes its OWN earlier statement, computation or interpretation is wrong and explicitly rejects and REPLACES it, in the segment that installs the replacement (even if the replacement is wrong). Rejecting an alternative it only just considered is reasoning. Merely recomputing is not correction.

answer_status (about the answer to the ORIGINAL question)
- none: no answer proposed: givens, intermediate quantities, evaluating or rejecting an option.
- candidate: proposes or endorses a specific answer, even tentatively; or a conclusion-sounding statement that is LATER followed by a doubt, check, correction or reopening; or a segment inside a check that restates the proposed answer.
- final: the model commits ("so the answer is...", "therefore...", Final Answer, \\boxed{{}}) and nothing after it doubts, re-checks or changes the answer. After the last doubt/check/correction, every segment that states the answer as the conclusion is final, including "So I think that's the answer." and the boxed line. A trace cut off before any commitment has no final.
Rule of thumb: scan FORWARD from a conclusion-sounding segment. If a doubt, verification or correction follows, it is candidate. If only restatements follow, it is final.

answer_value: a string or null. null when answer_status is none, or when the answer is long (code, text). Copy the answer in the segment's own form, do not normalize ("two" stays "two"). If the segment commits only by reference ("that's the answer"), copy the referent verbatim from the nearest earlier segment that wrote it, or null if no segment wrote it. Multiple choice: the option letter if named, otherwise the content as written. A correction that rejects 25 and installs 22 has value "22". Never write a value that no segment wrote.

temporal_role
- new: starts a different move: a new sub-goal, a new approach, or ANY change of discourse_role. The first segment of the trace is always new. A conclusion that follows a check or a computation chain is new.
- continuation: continues or completes the move in progress; it would read as an unfinished thought without the previous segment. A continuation ALWAYS has the same discourse_role as the previous segment: if the role changes, it is new. Items inside one announced plan (options A-D, test cases, each prime tested) are continuation once the plan has started.

Answer with JSON only: {{"segments": [...]}} with exactly one object per requested segment, in the requested order, with the keys: {OUT_KEYS}."""


def render_user(segs, question, lo, hi, clip=None):
    """The request for segs[lo:hi]. The whole trace comes first and the request last, so the server can reuse its
    prompt cache between the chunks of one trace. `clip` shortens only the segments that are NOT being labeled."""
    parts = []
    if question:
        parts.append("QUESTION:\n" + question.strip())
    lines = []
    for i, s in enumerate(segs):
        text = s["text"]
        if clip is not None and not (lo <= i < hi) and len(text) > clip:
            text = text[:clip] + " [...]"
        lines.append(f"{s['id']} {text}")
    parts.append(f"TRACE ({len(segs)} segments):\n\n" + "\n\n".join(lines))
    parts.append(f"Label these {hi - lo} segments, in this order: " + ", ".join(s["id"] for s in segs[lo:hi]))
    return "\n\n".join(parts)


# ---- the one worked example (a doubt -> correction -> verification -> final trace) ----------------------------------
EX_ROWS = [  # (segment text, note, discourse_role, answer_status, answer_value, temporal_role)
    ("I need to count the integers from 1 to 49 that are divisible by 3 or by 5.",
     "Restates the question.", "setup", "none", None, "new"),
    ("Multiples of 3 up to 49: floor(49/3) = 16.",
     "Starts solving: first count.", "reasoning", "none", None, "new"),
    ("Multiples of 5 up to 49: floor(49/5) = 9.",
     "Same plan: second count.", "reasoning", "none", None, "continuation"),
    ("Adding gives 16 + 9 = 25.",
     "Completes the plan; first answer to the question.", "reasoning", "candidate", "25", "continuation"),
    ("Hmm, numbers like 15 and 30 are multiples of both, so maybe I counted them twice.",
     "Suspects double counting, nothing checked yet.", "doubt", "none", None, "new"),
    ("Yes, 15, 30 and 45 are in both lists, so 25 overcounts; the count is 16 + 9 - 3 = 22.",
     "Rejects its own 25 and installs 22.", "correction", "candidate", "22", "new"),
    ("Check by listing: 13 multiples of only 3, 6 of only 5, and 3 of both gives 13 + 6 + 3 = 22.",
     "Independent check of 22.", "verification", "candidate", "22", "new"),
    ("So 22 integers satisfy the condition.",
     "Commits to 22 after the check; nothing reopens it.", "reasoning", "final", "22", "new"),
]
EX_SEGS = [{"id": f"⟦S{i + 1}⟧", "text": r[0]} for i, r in enumerate(EX_ROWS)]
EX_USER = render_user(EX_SEGS, None, 0, len(EX_SEGS))


def label_dict(sid, note, d, s, v, t):
    out = {"segment_id": sid}
    if CFG.USE_NOTE:
        out["note"] = note
    out.update(discourse_role=d, answer_status=s, answer_value=v, temporal_role=t)
    return out


EX_ASSISTANT = json.dumps({"segments": [label_dict(EX_SEGS[i]["id"], *r[1:]) for i, r in enumerate(EX_ROWS)]}, ensure_ascii=False)

PROMPT_CHARS = len(SYSTEM_PROMPT) + len(EX_USER) + len(EX_ASSISTANT)
FINGERPRINT = md5(json.dumps([SYSTEM_PROMPT, EX_USER, EX_ASSISTANT, os.path.basename(GGUF_PATH), CFG.USE_NOTE,
                              CFG.NOTE_MAX_CHARS, CFG.CHUNK_SIZE, CFG.ENFORCE_RULES]))
print(f"System prompt {len(SYSTEM_PROMPT):,} chars + worked example {len(EX_USER) + len(EX_ASSISTANT):,} chars  (~{PROMPT_CHARS // 3:,} tokens in total)")
print("Run fingerprint:", FINGERPRINT[:12])
print("\n--- worked example, as the model sees it ---\n" + EX_USER + "\n\n--- expected answer ---\n" + EX_ASSISTANT[:600] + " ...")

System prompt 4,436 chars + worked example 2,064 chars  (~2,166 tokens in total)
Run fingerprint: 59d91aeb4a5a

--- worked example, as the model sees it ---
TRACE (8 segments):

⟦S1⟧ I need to count the integers from 1 to 49 that are divisible by 3 or by 5.

⟦S2⟧ Multiples of 3 up to 49: floor(49/3) = 16.

⟦S3⟧ Multiples of 5 up to 49: floor(49/5) = 9.

⟦S4⟧ Adding gives 16 + 9 = 25.

⟦S5⟧ Hmm, numbers like 15 and 30 are multiples of both, so maybe I counted them twice.

⟦S6⟧ Yes, 15, 30 and 45 are in both lists, so 25 overcounts; the count is 16 + 9 - 3 = 22.

⟦S7⟧ Check by listing: 13 multiples of only 3, 6 of only 5, and 3 of both gives 13 + 6 + 3 = 22.

⟦S8⟧ So 22 integers satisfy the condition.

Label these 8 segments, in this order: ⟦S1⟧, ⟦S2⟧, ⟦S3⟧, ⟦S4⟧, ⟦S5⟧, ⟦S6⟧, ⟦S7⟧, ⟦S8⟧

--- expected answer ---
{"segments": [{"segment_id": "⟦S1⟧", "note": "Restates the question.", "discourse_role": "setup", "answer_status": "none", "answer_value": null, "temporal_role": "new"}, {"segment

In [8]:
# 7. Labeling logic: one HTTP call per chunk of segments, retries, consistency fixes
DISCOURSE_ROLES = ["setup", "reasoning", "doubt", "verification", "correction"]
ANSWER_STATUSES = ["none", "candidate", "final"]
TEMPORAL_ROLES = ["new", "continuation"]
NOTE_TOKENS = (CFG.NOTE_MAX_CHARS or 300) // 3 if CFG.USE_NOTE else 0

STATS, STATS_LOCK, STOP = Counter(), threading.Lock(), threading.Event()
TLS = threading.local()        # every worker thread (cell 11) is pinned to ONE server = ONE GPU through TLS.url
GPU_STATS = {}                 # gpu index -> Counter(requests, prompt_tokens, completion_tokens), filled by call_llm (same lock as STATS)
RUN = {}                       # bookkeeping of the last full run (cell 11), read by throughput_report()


def current_url():
    """URL of this thread's server. The main thread (cell 10) uses the first GPU. SERVERS is created in cell 9."""
    return getattr(TLS, "url", None) or SERVERS[0]["url"]


def make_schema(n):
    """JSON schema for the answer; llama-server turns it into a grammar, so the output is always valid JSON with allowed values."""
    props = {"segment_id": {"type": "string"}}
    if CFG.USE_NOTE:
        props["note"] = {"type": "string", **({"maxLength": CFG.NOTE_MAX_CHARS} if CFG.NOTE_MAX_CHARS else {})}
    props["discourse_role"] = {"type": "string", "enum": DISCOURSE_ROLES}
    props["answer_status"] = {"type": "string", "enum": ANSWER_STATUSES}
    props["answer_value"] = {"type": ["string", "null"]}
    props["temporal_role"] = {"type": "string", "enum": TEMPORAL_ROLES}
    item = {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}
    return {"type": "object", "required": ["segments"], "additionalProperties": False,
            "properties": {"segments": {"type": "array", "items": item, "minItems": n, "maxItems": n}}}


def call_llm(user_msg, n, temperature, seed):
    """System prompt, then the worked example as a user/assistant turn, then the real request."""
    payload = {"model": "labeler", "stream": False, "temperature": temperature, "seed": seed,
               "max_tokens": n * (80 + NOTE_TOKENS) + 100, "cache_prompt": True,
               "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": EX_USER},
                            {"role": "assistant", "content": EX_ASSISTANT},
                            {"role": "user", "content": user_msg}],
               "response_format": {"type": "json_schema", "json_schema": {"name": "labels", "strict": True, "schema": make_schema(n)}},
               "chat_template_kwargs": {"enable_thinking": False}}      # the JSON grammar cannot contain a thinking block
    url = current_url()
    r = requests.post(url + "/v1/chat/completions", json=payload, timeout=CFG.REQUEST_TIMEOUT_S)
    r.raise_for_status()
    data = r.json()
    choice = data["choices"][0]
    if choice.get("finish_reason") == "length":
        raise ValueError("answer was cut at max_tokens")
    usage = data.get("usage") or {}
    with STATS_LOCK:
        STATS["requests"] += 1
        STATS["prompt_tokens"] += usage.get("prompt_tokens") or 0
        STATS["completion_tokens"] += usage.get("completion_tokens") or 0
        _g = GPU_STATS.setdefault(next((s["gpu"] for s in SERVERS if s["url"] == url), None), Counter())      # same numbers, per GPU
        _g["requests"] += 1
        _g["prompt_tokens"] += usage.get("prompt_tokens") or 0
        _g["completion_tokens"] += usage.get("completion_tokens") or 0
    return choice["message"]["content"] or ""


def check_labels(content, ids):
    """Parse the answer and make sure it holds exactly one valid label per requested segment, in order."""
    items = json.loads(content)["segments"]
    if len(items) != len(ids):
        raise ValueError(f"expected {len(ids)} labels, got {len(items)}")
    for it, sid in zip(items, ids):
        if it["segment_id"] != sid:
            raise ValueError(f"expected {sid}, got {it['segment_id']}")
        if it["discourse_role"] not in DISCOURSE_ROLES or it["answer_status"] not in ANSWER_STATUSES or it["temporal_role"] not in TEMPORAL_ROLES:
            raise ValueError(f"invalid label value for {sid}")
        if it["answer_value"] is not None and not isinstance(it["answer_value"], str):
            raise ValueError(f"answer_value of {sid} is not a string")
    return items


def label_chunk(trace, lo, hi):
    segs = trace["segments"]
    ids = [s["id"] for s in segs[lo:hi]]
    n = hi - lo
    # fit the prompt in one slot: only the segments that are NOT being labeled get shortened, and only if needed
    max_chars = (CFG.CTX_PER_SLOT - (n * (80 + NOTE_TOKENS) + 100) - 300) * 3 - PROMPT_CHARS      # ~3 characters per token (conservative)
    user = next((u for u in (render_user(segs, trace["question"], lo, hi, clip) for clip in (None, 600, 300, 150, 60))
                 if len(u) <= max_chars), None)
    if user is None:
        raise ValueError(f"{ids[0]}..{ids[-1]} do not fit in CFG.CTX_PER_SLOT={CFG.CTX_PER_SLOT}: raise it or lower CHUNK_SIZE")
    last = None
    for attempt in range(CFG.MAX_RETRIES + 1):
        if STOP.is_set():
            raise RuntimeError("stopped")
        try:
            return check_labels(call_llm(user, n, CFG.TEMPERATURE if attempt == 0 else CFG.RETRY_TEMPERATURE, CFG.SEED + attempt), ids)
        except (requests.RequestException, ValueError, KeyError, TypeError) as e:     # ValueError includes bad JSON
            last = f"{type(e).__name__}: {str(e)[:200]}"
            with STATS_LOCK:
                STATS["retries"] += 1
            if isinstance(e, requests.RequestException):
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"{ids[0]}..{ids[-1]} failed after {CFG.MAX_RETRIES + 1} attempts: {last}")


def enforce_rules(labels):
    """Cheap fixes for things the labeling rules forbid. Returns how many fields were changed."""
    fixes = 0

    def fix(lab, key, value):
        nonlocal fixes
        if lab[key] != value:
            lab[key] = value
            fixes += 1

    for i, lab in enumerate(labels):
        if lab["answer_status"] == "none" or not (lab["answer_value"] or "").strip():
            fix(lab, "answer_value", None)                     # no answer -> no value
        if i == 0:
            fix(lab, "temporal_role", "new")                   # the first segment starts a move
        elif lab["temporal_role"] == "continuation" and lab["discourse_role"] != labels[i - 1]["discourse_role"]:
            fix(lab, "temporal_role", "new")                   # a continuation never changes the role
    checks = [i for i, lab in enumerate(labels) if lab["discourse_role"] in ("doubt", "verification", "correction")]
    last_check = checks[-1] if checks else -1
    for lab in labels[:max(last_check, 0)]:
        if lab["answer_status"] == "final":
            fix(lab, "answer_status", "candidate")             # a conclusion that is later doubted/checked was only a candidate
    return fixes


def label_trace(trace):
    t0 = time.time()
    n = len(trace["segments"])
    labels = []
    for lo in range(0, n, CFG.CHUNK_SIZE):
        labels += label_chunk(trace, lo, min(lo + CFG.CHUNK_SIZE, n))
    fixes = enforce_rules(labels) if CFG.ENFORCE_RULES else 0
    return {"key": trace["key"], "fp": FINGERPRINT, "row_id": trace["row_id"], "n": n, "labels": labels,
            "fixes": fixes, "seconds": round(time.time() - t0, 1)}


# ---- throughput report (used by cells 11 and 12; built only from counters we already keep, no HTTP calls) ----------------
def snapshot_counters():
    with STATS_LOCK:
        return Counter(STATS), {g: Counter(c) for g, c in GPU_STATS.items()}


def _duration(seconds):
    m, s = divmod(int(round(seconds)), 60)
    h, m = divmod(m, 60)
    return f"{h} h {m:02d} min {s:02d} s" if h else f"{m} min {s:02d} s"


def throughput_report():
    """Speed of the LAST full run (cell 11): wall time, traces/min, tokens/s overall and per GPU.
    Prompt tokens are the prompt sizes the server reports, so they include any prefix it reused from its prompt cache."""
    if not RUN:
        print("Throughput: no full run in this session yet (run cell 11).")
        return
    wall = max((RUN.get("t1") or time.time()) - RUN["t0"], 1e-9)
    now_stats, now_gpu = snapshot_counters()
    tot = now_stats - RUN["stats0"]                                           # only what THIS run added
    ok, n_failed = RUN["ok"], len(FAILED)
    n_ok = sum(ok.values())
    print("=" * 100)
    print(f"THROUGHPUT of the last full run   ({len(SERVERS)} GPU x SERVER_SLOTS={CFG.SERVER_SLOTS} = {len(SERVERS) * CFG.SERVER_SLOTS} parallel requests, CTX_PER_SLOT={CFG.CTX_PER_SLOT:,})")
    print(f"  wall time        {_duration(wall)}  ({wall:,.0f} s)")
    print(f"  traces labeled   {n_ok:,}" + (f"   (+{n_failed} failed)" if n_failed else "") + f"   ->  {60 * n_ok / wall:.2f} traces/min")
    print(f"  model calls      {tot['requests']:,}   retries {tot['retries']:,}")
    print(f"  tokens/s overall prompt {tot['prompt_tokens'] / wall:,.0f}   completion {tot['completion_tokens'] / wall:,.1f}"
          f"      (totals: prompt {tot['prompt_tokens']:,}, completion {tot['completion_tokens']:,})")
    for g in sorted({s["gpu"] for s in SERVERS} | set(now_gpu) | set(ok), key=str):
        d = now_gpu.get(g, Counter()) - RUN["gpu0"].get(g, Counter())
        print(f"  GPU {g}: {ok.get(g, 0):>5,} traces ({60 * ok.get(g, 0) / wall:5.2f}/min)   prompt {d['prompt_tokens'] / wall:>8,.0f} tok/s"
              f"   completion {d['completion_tokens'] / wall:>7,.1f} tok/s   {d['requests']:,} calls")
    print("=" * 100)


In [9]:
# 8. Checkpoint (one line per finished trace) and CSV export
CKPT_PATH = os.path.join(CFG.OUTPUT_DIR, CFG.OUT_CHECKPOINT)
DONE = {}      # trace key -> record, only traces finished with the CURRENT prompt/model/settings
FAILED = {}    # row_id -> reason, this session only (failed traces are simply tried again on the next run)


def load_checkpoint():
    DONE.clear()
    lines = stale = 0
    if os.path.isfile(CKPT_PATH):
        for line in open(CKPT_PATH, encoding="utf-8"):
            try:
                rec = json.loads(line)
            except ValueError:
                continue                                      # half-written last line after a crash
            lines += 1
            if rec.get("fp") == FINGERPRINT:
                DONE[rec["key"]] = rec
            else:
                stale += 1                                    # other prompt / model / settings / first version: not reused
    return lines, stale


def save_checkpoint(rec):
    with open(CKPT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
    DONE[rec["key"]] = rec


LABEL_KEYS = ["segment_id", "discourse_role", "answer_status", "answer_value", "temporal_role"]
SEG_COLS = ID_COLS + ["row_id", "position", "segment_id", "segment_text", "discourse_role", "answer_status",
                      "answer_value", "temporal_role", "note"]


def _write_csv(df, name):
    path = os.path.join(CFG.OUTPUT_DIR, name)
    df.to_csv(path + ".tmp", index=False, encoding=CFG.CSV_ENCODING)
    os.replace(path + ".tmp", path)                            # a crash can never leave a half-written CSV


def export_csvs():
    """Rewrite both CSV files from what is finished so far. Only ever called from the main thread."""
    by_row = {t["row_id"]: t for t in TRACES}
    status, n_segs, labels_json, seg_rows = [], [], [], []
    for row_id in range(len(DF)):
        t = by_row.get(row_id)
        rec = DONE.get(t["key"]) if t else None
        n_segs.append(len(t["segments"]) if t else 0)
        if t is None:
            status.append("skipped: " + SKIPPED.get(row_id, "?"))
        elif rec is None:
            status.append("failed: " + FAILED[row_id] if row_id in FAILED else "pending")
        else:
            status.append("labeled")
        if rec is None:
            labels_json.append("")
            continue
        labels_json.append(json.dumps([{k: lab[k] for k in LABEL_KEYS} for lab in rec["labels"]], ensure_ascii=False, separators=(",", ":")))
        for pos, (seg, lab) in enumerate(zip(t["segments"], rec["labels"])):
            seg_rows.append({**t["ids"], "row_id": row_id, "position": pos, "segment_id": seg["id"], "segment_text": seg["text"],
                             "discourse_role": lab["discourse_role"], "answer_status": lab["answer_status"],
                             "answer_value": "null" if lab["answer_value"] is None else lab["answer_value"],
                             "temporal_role": lab["temporal_role"], "note": lab.get("note", "")})
    res = DF.copy()
    res["rtl_status"], res["rtl_n_segments"], res["rtl_labels_json"] = status, n_segs, labels_json
    _write_csv(res, CFG.OUT_RESULTS)
    _write_csv(pd.DataFrame(seg_rows, columns=SEG_COLS), CFG.OUT_SEGMENTS)


_lines, _stale = load_checkpoint()
_reusable = sum(1 for t in TRACES if t["key"] in DONE)
print(f"Checkpoint {CKPT_PATH}: {_lines} lines, {_reusable} traces already labeled and reusable, {_stale} lines from another prompt/model (ignored)")

Checkpoint /kaggle/working/labeling_checkpoint.jsonl: 1314 lines, 1314 traces already labeled and reusable, 0 lines from another prompt/model (ignored)


In [10]:
# 9. Start one llama-server PER GPU (data parallelism: each T4 holds a full copy of the model)
import atexit, signal

if CFG.KV_CACHE_TYPE and not CFG.FLASH_ATTN:
    raise ValueError("A quantized KV cache needs FLASH_ATTN=True (or set KV_CACHE_TYPE=None).")

_old_procs = {s["port"]: s["proc"] for s in globals().get("SERVERS", [])}      # keep handles when this cell is re-run
SERVERS = [dict(gpu=g, port=CFG.PORT_BASE + k, url=f"http://127.0.0.1:{CFG.PORT_BASE + k}",
                log=os.path.join(CFG.WORK_DIR, f"llama-server-gpu{g}.log"), proc=_old_procs.get(CFG.PORT_BASE + k))
           for k, g in enumerate(CFG.GPU_INDICES)]


def server_ready(s):
    try:
        return requests.get(s["url"] + "/health", timeout=3).status_code == 200
    except requests.RequestException:
        return False


def server_dead(s):
    """True only when we started this process ourselves and it has exited (no HTTP call, so it is safe under load)."""
    return s["proc"] is not None and s["proc"].poll() is not None


def stop_server(s):
    p = s["proc"]
    if p is not None and p.poll() is None:
        try:
            os.killpg(p.pid, signal.SIGTERM)
            p.wait(timeout=30)
        except Exception:
            os.killpg(p.pid, signal.SIGKILL)
    s["proc"] = None


def stop_servers():
    for s in SERVERS:
        stop_server(s)


def start_servers(timeout_s=1200):
    todo = []
    for s in SERVERS:
        if server_ready(s):
            print(f"GPU {s['gpu']}: llama-server already running on port {s['port']} (call stop_servers() first if you changed a server setting).")
        else:
            todo.append(s)
    for s in todo:                                                # start all first, then wait: the models load in parallel
        cmd = [LLAMA_SERVER_BIN, "-m", GGUF_PATH, "--alias", "labeler", "--host", "127.0.0.1", "--port", str(s["port"]),
               "-c", str(CFG.SERVER_SLOTS * CFG.CTX_PER_SLOT), "--parallel", str(CFG.SERVER_SLOTS),
               "-b", str(CFG.BATCH_SIZE), "--ubatch-size", str(CFG.UBATCH_SIZE), "-ngl", "999", "--jinja"]
        if CFG.FLASH_ATTN:
            cmd += ["-fa", "on"]
        if CFG.KV_CACHE_TYPE:
            cmd += ["-ctk", CFG.KV_CACHE_TYPE, "-ctv", CFG.KV_CACHE_TYPE]
        print(f"GPU {s['gpu']}: starting", " ".join(cmd))
        s["proc"] = subprocess.Popen(cmd, stdout=open(s["log"], "w"), stderr=subprocess.STDOUT, start_new_session=True,
                                     env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(s["gpu"])))
    t0 = time.time()
    while todo:
        for s in list(todo):
            if server_ready(s):
                print(f"GPU {s['gpu']}: server ready after {time.time() - t0:.0f} s")
                todo.remove(s)
            elif s["proc"].poll() is not None:
                tail = "".join(open(s["log"]).readlines()[-40:])
                stop_servers()
                raise RuntimeError(f"llama-server for GPU {s['gpu']} exited ({s['proc'] and s['proc'].returncode}). Last log lines ({s['log']}):\n{tail}")
        if todo and time.time() - t0 > timeout_s:
            stop_servers()
            raise TimeoutError(f"llama-server not ready after {timeout_s} s; see {[s['log'] for s in todo]}")
        if todo:
            time.sleep(2)


atexit.register(stop_servers)


def gpu_memory_gib(gpu):
    """(used, free, total) in GiB from nvidia-smi (it reports MiB), or None when it cannot be read."""
    r = sh(["nvidia-smi", "-i", str(gpu), "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader,nounits"])
    try:
        used, free, total = (float(x) / 1024 for x in r.stdout.strip().splitlines()[0].split(","))
        return used, free, total
    except (ValueError, IndexError):
        return None


def report_vram():
    """VRAM per GPU once the servers are up. The KV cache is allocated at start-up, so this is what is really left."""
    print(f"Server settings: SERVER_SLOTS={CFG.SERVER_SLOTS} x CTX_PER_SLOT={CFG.CTX_PER_SLOT:,} = {CFG.SERVER_SLOTS * CFG.CTX_PER_SLOT:,} tokens of context per GPU")
    for s in SERVERS:
        m = gpu_memory_gib(s["gpu"])
        if m is None:
            print(f"GPU {s['gpu']} VRAM: unknown (nvidia-smi gave no usable answer)   (port {s['port']}, log {s['log']})")
            continue
        used, free, total = m
        print(f"GPU {s['gpu']} VRAM: used {used:.2f} GiB, free {free:.2f} GiB, total {total:.2f} GiB   (port {s['port']}, log {s['log']})")
        if free < CFG.VRAM_WARN_GIB:
            print(f"  WARNING: GPU {s['gpu']} has only {free:.2f} GiB free (< {CFG.VRAM_WARN_GIB:g} GiB). A long prompt can run out of memory in the middle of the run. "
                  f"Lower CFG.SERVER_SLOTS (now {CFG.SERVER_SLOTS}) or CFG.CTX_PER_SLOT, then run stop_servers() and this cell again.")


start_servers()
report_vram()

GPU 0: starting /kaggle/temp/llama.cpp/build/bin/llama-server -m /kaggle/temp/models/Qwen3.5-9B-Q4_K_M.gguf --alias labeler --host 127.0.0.1 --port 8089 -c 98304 --parallel 6 -b 2048 --ubatch-size 512 -ngl 999 --jinja -fa on -ctk q8_0 -ctv q8_0
GPU 1: starting /kaggle/temp/llama.cpp/build/bin/llama-server -m /kaggle/temp/models/Qwen3.5-9B-Q4_K_M.gguf --alias labeler --host 127.0.0.1 --port 8090 -c 98304 --parallel 6 -b 2048 --ubatch-size 512 -ngl 999 --jinja -fa on -ctk q8_0 -ctv q8_0
GPU 0: server ready after 6 s
GPU 1: server ready after 6 s
Server settings: SERVER_SLOTS=6 x CTX_PER_SLOT=16,384 = 98,304 tokens of context per GPU
GPU 0 VRAM: used 7.21 GiB, free 7.35 GiB, total 15.00 GiB   (port 8089, log /kaggle/temp/llama-server-gpu0.log)
GPU 1 VRAM: used 7.21 GiB, free 7.35 GiB, total 15.00 GiB   (port 8090, log /kaggle/temp/llama-server-gpu1.log)


In [ ]:
# 10. Full run, data parallel: every GPU has its own server, every server has CFG.SERVER_SLOTS worker threads.
# All workers take the next trace from ONE shared queue. A checkpoint line is written after EVERY trace;
# both CSV files are rewritten every CFG.AUTOSAVE_SECONDS (5 min). Only this (main) thread writes files.
import queue

load_checkpoint()                                    # re-read the file, so re-running this cell after an interruption resumes
PENDING = [t for t in TRACES if t["key"] not in DONE]
FAILED.clear()
n_workers = len(SERVERS) * CFG.SERVER_SLOTS
print(f"{len(TRACES) - len(PENDING)} traces already done, {len(PENDING)} to label, on {len(SERVERS)} GPU(s) x {CFG.SERVER_SLOTS} slots = {n_workers} parallel requests.")

JOBS, RESULTS = queue.Queue(), queue.Queue()
for t in PENDING:
    JOBS.put(t)


def worker(srv):
    TLS.url = srv["url"]                             # this thread only ever talks to its own GPU
    while not STOP.is_set():
        if server_dead(srv):
            break                                    # this GPU's server died: the other GPU(s) take the remaining traces
        try:
            t = JOBS.get_nowait()
        except queue.Empty:
            break
        try:
            RESULTS.put((t, label_trace(t), None, srv))
        except Exception as e:
            if server_dead(srv):
                JOBS.put(t)                          # not this trace's fault: give it to another GPU
                RESULTS.put((None, None, f"llama-server on GPU {srv['gpu']} died, see {srv['log']}", srv))
                break
            RESULTS.put((t, None, f"{type(e).__name__}: {e}"[:300], srv))     # one bad trace never stops the run


STOP.clear()
PER_GPU = Counter()                                  # traces finished per GPU, successes AND failures (progress bar)
OK_PER_GPU = Counter()                               # traces labeled and checkpointed per GPU (throughput report)
threads = [threading.Thread(target=worker, args=(srv,), daemon=True, name=f"gpu{srv['gpu']}-{k}")
           for srv in SERVERS for k in range(CFG.SERVER_SLOTS)]
last_save = time.time()
bar = tqdm(total=len(PENDING), unit="trace")
_stats0, _gpu0 = snapshot_counters()                 # baseline: the report counts only what THIS run adds (cell 10 used the servers too)
RUN.clear()
RUN.update(t0=time.time(), t1=None, stats0=_stats0, gpu0=_gpu0, ok=OK_PER_GPU)
try:
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads) or not RESULTS.empty():
        try:
            trace, rec, err, srv = RESULTS.get(timeout=10)
        except queue.Empty:
            trace = rec = err = srv = None
        if srv is not None:
            if trace is None:
                tqdm.write("WARNING: " + err)
            else:
                if rec is not None:
                    save_checkpoint(rec)
                else:
                    FAILED[trace["row_id"]] = err
                PER_GPU[srv["gpu"]] += 1
                if rec is not None:
                    OK_PER_GPU[srv["gpu"]] += 1
                bar.update(1)
                bar.set_postfix(failed=len(FAILED), retries=STATS["retries"], **{f"gpu{g}": n for g, n in sorted(PER_GPU.items())})
        if time.time() - last_save >= CFG.AUTOSAVE_SECONDS:
            export_csvs()
            last_save = time.time()
            tqdm.write(f"[{time.strftime('%H:%M:%S')}] CSV files saved ({sum(t['key'] in DONE for t in TRACES)}/{len(TRACES)} traces labeled)")
finally:                                             # also runs when you stop the cell: nothing finished is ever lost
    STOP.set()
    RUN["t1"] = time.time()                          # wall time stops here, before the CSV export
    bar.close()
    export_csvs()
    print("Traces finished per GPU in this run:", dict(sorted(PER_GPU.items())))
    print("CSV files saved:", os.path.join(CFG.OUTPUT_DIR, CFG.OUT_RESULTS), "and", os.path.join(CFG.OUTPUT_DIR, CFG.OUT_SEGMENTS))
    throughput_report()


1314 traces already done, 2731 to label, on 2 GPU(s) x 6 slots = 12 parallel requests.


  0%|          | 0/2731 [00:00<?, ?trace/s]

In [ ]:
# 12. Summary
recs = [DONE[t["key"]] for t in TRACES if t["key"] in DONE]
labs = [lab for r in recs for lab in r["labels"]]
print(f"Traces: {len(recs)} labeled, {len(FAILED)} failed, {len(TRACES) - len(recs) - len(FAILED)} pending, {len(SKIPPED)} rows skipped")
print(f"Segments labeled: {len(labs):,}   enforce_rules changed {sum(r['fixes'] for r in recs)} fields   "
      f"seconds per trace: median {statistics.median([r['seconds'] for r in recs]) if recs else 0:.0f}")
for field in ("discourse_role", "answer_status", "temporal_role"):
    c = Counter(l[field] for l in labs)
    print(f"  {field:<15}", "  ".join(f"{k} {v} ({100 * v / max(len(labs), 1):.0f}%)" for k, v in c.most_common()))
print(f"  traces with at least one 'final': {sum(any(l['answer_status'] == 'final' for l in r['labels']) for r in recs)}/{len(recs)}")
print("This session's model calls:", dict(STATS))
throughput_report()                                  # speed of the last full run (cell 11)
for row_id, reason in list(FAILED.items())[:10]:
    print(f"  FAILED row {row_id}: {reason}")
if FAILED or len(recs) < len(TRACES):
    print("Run cell 11 again to retry the failed / pending traces; finished ones are skipped.")
else:
    stop_servers()                                   # everything is labeled: free both GPUs
for name in (CFG.OUT_RESULTS, CFG.OUT_SEGMENTS, CFG.OUT_CHECKPOINT):
    p = os.path.join(CFG.OUTPUT_DIR, name)
    print(f"  {p}  ({os.path.getsize(p):,} bytes)" if os.path.isfile(p) else f"  {p}  (not written yet)")